# Notebook 03k — SHAP Values (target: TUMOR)

**Objetivo**: explicar qué variables empujan la predicción de fusiones teloméricas en cada paciente.  
**Modelo**: Lasso con alpha óptimo (0.004642, encontrado en 03j con Optuna).  
**Features**: mismas que 03c (verdes + azules + mesenquimal).  
**SHAP**: calcula la contribución exacta de cada variable a la predicción de cada paciente.

## 1. Imports

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'shap', '-q'], check=True)

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import shap

from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print(f'CWD: {os.getcwd()}')
print(f'SHAP version: {shap.__version__}')
print('Librerías cargadas OK')

## 2. Carga de datos (idéntica a 03c)

In [ ]:
TARGET = 'Tumor_Telomere fusion rate (Gt-end/coverage*purity)'

df = pd.read_csv('datos_limpios_modelo_a.csv')

df_verde = pd.read_excel('revision_color_code.xlsx', sheet_name='Sheet1',
                         usecols=['Patient_ID_IF_mod', 'firstResponse', 'treatment'])
df = df.merge(df_verde, on='Patient_ID_IF_mod', how='left')

df_azul = pd.read_excel('revision_color_code.xlsx', sheet_name='Sheet1',
                        usecols=['Patient_ID_IF_mod', 'biopsySite', 'hasSystemicPreTreatment',
                                 'hasRadiotherapyPreTreatment', 'treatmentGiven', 'treatment1'])
df = df.merge(df_azul, on='Patient_ID_IF_mod', how='left')

df_mes = pd.read_excel('Tumor_mesenchimal.xlsx')
df_mes.columns = ['cancer_type_code', 'Mesenchimal_origin']
df = df.merge(df_mes, on='cancer_type_code', how='left')

print(f'Shape final: {df.shape}')

## 3. Target y features (idénticos a 03c)

In [ ]:
y = np.log1p(df[TARGET])

df['firstResponse'] = df['firstResponse'].fillna('Sin_dato')
df['Mesenchimal_origin'] = df['Mesenchimal_origin'].fillna('Sin_dato')
top15 = df['treatment'].value_counts().head(15).index.tolist()
df['treatment_agrupado'] = df['treatment'].apply(
    lambda x: x if x in top15 else ('Sin_dato' if pd.isna(x) else 'Otro')
)
df['hasSystemicPreTreatment'] = df['hasSystemicPreTreatment'].fillna('Sin_dato')
df['hasRadiotherapyPreTreatment'] = df['hasRadiotherapyPreTreatment'].fillna('Sin_dato')
df['treatmentGiven'] = df['treatmentGiven'].fillna('Sin_dato')
top10_biopsy = df['biopsySite'].value_counts().head(10).index.tolist()
df['biopsySite_agrupado'] = df['biopsySite'].apply(
    lambda x: x if x in top10_biopsy else ('Sin_dato' if pd.isna(x) else 'Otro')
)
top10_t1 = df['treatment1'].value_counts().head(10).index.tolist()
df['treatment1_agrupado'] = df['treatment1'].apply(
    lambda x: x if x in top10_t1 else ('Sin_dato' if pd.isna(x) else 'Otro')
)

cols_features = ['firstResponse', 'treatment_agrupado', 'Mesenchimal_origin',
                 'cancer_type', 'biopsySite_agrupado', 'hasSystemicPreTreatment',
                 'hasRadiotherapyPreTreatment', 'treatmentGiven', 'treatment1_agrupado']

X = pd.get_dummies(df[cols_features], drop_first=True)
print(f'Shape X: {X.shape} | Shape y: {y.shape}')

## 4. Entrenar Lasso con alpha óptimo (03j Optuna)

Usamos alpha=0.004642 — el mejor valor encontrado por Optuna en 03j (R² CV=0.0617).
Entrenamos sobre el dataset completo para que SHAP vea todos los pacientes.

In [ ]:
ALPHA_OPTIMO = 0.004642

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# DataFrame escalado con nombres de columnas — necesario para que SHAP muestre los nombres
X_train_sc_df = pd.DataFrame(X_train_sc, columns=X.columns)
X_test_sc_df  = pd.DataFrame(X_test_sc,  columns=X.columns)

modelo = Lasso(alpha=ALPHA_OPTIMO, max_iter=10000, random_state=42)
modelo.fit(X_train_sc_df, y_train)

from sklearn.metrics import r2_score
r2 = r2_score(y_test, modelo.predict(X_test_sc_df))
n_nonzero = (modelo.coef_ != 0).sum()
print(f'R² test: {r2:.4f}')
print(f'Coeficientes no nulos: {n_nonzero} / {len(modelo.coef_)}')

## 5. Calcular SHAP values

Para modelos lineales se usa `LinearExplainer` — es exacto y rápido.
Calculamos los SHAP values sobre el conjunto de test (los pacientes que el modelo no vio).

In [ ]:
explainer   = shap.LinearExplainer(modelo, X_train_sc_df)
shap_values = explainer(X_test_sc_df)

print(f'Shape SHAP values: {shap_values.values.shape}')
print(f'  -> {shap_values.values.shape[0]} pacientes x {shap_values.values.shape[1]} features')
print('SHAP calculado OK')

## 6. Gráfico 1 — Importancia global (bar plot)

Muestra las 20 variables más importantes en promedio (media del valor absoluto de SHAP).
No distingue positivo/negativo — solo cuánto influye cada variable en la predicción.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
shap.plots.bar(shap_values, max_display=20, show=False, ax=ax)
ax.set_title('Importancia global — SHAP (Lasso, target: TUMOR)', fontsize=12)
plt.tight_layout()
plt.savefig('notebooks/img/img_03k_shap_bar.png', dpi=150, bbox_inches='tight')
plt.close()
print('Guardado: notebooks/img/img_03k_shap_bar.png')

## 7. Gráfico 2 — Beeswarm

El gráfico más informativo de SHAP. Cada punto es un paciente.
- Eje X: valor SHAP (positivo = empuja la predicción hacia MÁS fusiones)
- Color: valor de la feature (rojo = valor alto, azul = valor bajo)
- Posición vertical: variable ordenada por importancia

Permite ver tanto la importancia como la dirección del efecto de cada variable.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
shap.plots.beeswarm(shap_values, max_display=20, show=False)
plt.title('SHAP Beeswarm — Lasso (target: TUMOR)', fontsize=12)
plt.tight_layout()
plt.savefig('notebooks/img/img_03k_shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.close()
print('Guardado: notebooks/img/img_03k_shap_beeswarm.png')

## 8. Gráfico 3 — Waterfall de un paciente concreto

Explica la predicción de un paciente individual paso a paso.
Empieza desde la predicción media (baseline) y suma/resta la contribución de cada variable
hasta llegar a la predicción final de ese paciente.

Usamos el paciente del test con la prediccion mas alta (el que mas fusiones predice el modelo).

In [ ]:
# Paciente con prediccion mas alta en el test
y_pred_test = modelo.predict(X_test_sc_df)
idx_max     = np.argmax(y_pred_test)

print(f'Paciente seleccionado: indice {idx_max} en test')
print(f'  Prediccion (log1p): {y_pred_test[idx_max]:.4f}')
print(f'  Real      (log1p): {y_test.values[idx_max]:.4f}')

fig, ax = plt.subplots(figsize=(10, 7))
shap.plots.waterfall(shap_values[idx_max], max_display=15, show=False)
plt.title(f'SHAP Waterfall — paciente con mayor prediccion (idx={idx_max})', fontsize=11)
plt.tight_layout()
plt.savefig('notebooks/img/img_03k_shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.close()
print('Guardado: notebooks/img/img_03k_shap_waterfall.png')

## 9. Top variables por contribucion media firmada

Muestra las 15 variables con mayor SHAP medio (con signo).
Positivo = la variable empuja hacia MAS fusiones en promedio.
Negativo = la variable empuja hacia MENOS fusiones en promedio.

In [ ]:
shap_mean = pd.Series(shap_values.values.mean(axis=0), index=X.columns)
shap_abs  = shap_mean.abs().sort_values(ascending=False)

sep = '-'
print('=== TOP 15 — Contribucion media SHAP (con signo) ===')
print(f'  {"Variable":<50}  {"SHAP medio":>10}')
print(f'  {sep*50}  {sep*10}')
for var in shap_abs.head(15).index:
    val = shap_mean[var]
    direccion = 'MAS fusiones' if val > 0 else 'MENOS fusiones'
    print(f'  {var:<50}  {val:>+10.5f}  ({direccion})')